# SAR-Optical Fusion for Construction & Demolition Waste (CDW) Mapping

**Author:** Nikos Kordalis (nikos.kordalis@iccs.gr)

Binary classification of CDW deposits using fused Sentinel-1 SAR and Sentinel-2 optical data.

**Pipeline:**
1. ARD preprocessing with openEO (Sentinel-2 optical + spectral indices)
2. Multi-band composite creation (8 bands: VV, VH, B2, B3, B4, B8, NDVI, SAVI)
3. Pixel extraction from labelled polygons
4. ML classification (KNN, Random Forest) with cross-validation
5. Full-image prediction and GeoTIFF export

---
## Part 1: Analysis Ready Data (ARD) Preprocessing with openEO

Processes Sentinel-2 L2A data via openEO: cloud masking (SCL), temporal composites, NDVI and SAVI.

> **Note:** Sentinel-1 VV/VH bands are preprocessed separately and merged with the optical composite to create the final 8-band input image.

In [ ]:
import openeo

con = openeo.connect("openeo.dataspace.copernicus.eu")
con.authenticate_oidc()

### Configuration

In [ ]:
SPATIAL_EXTENT = {"west": 0.914, "south": 41.047, "east": 1.195, "north": 41.260}
TEMPORAL_EXTENT = ["2019-01-10", "2019-01-13"]
MAX_CLOUD_COVER = 25
OUTPUT_NAME = "S2_B_R_G_NIR_NDVI_SAVI_Jan_2019"

sentinel2_cube = con.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=SPATIAL_EXTENT,
    temporal_extent=TEMPORAL_EXTENT,
    bands=["B02", "B03", "B04", "B08", "SCL"],
    max_cloud_cover=MAX_CLOUD_COVER
)

# Rescale reflectance bands
blue = sentinel2_cube.band("B02") * 0.0001
green = sentinel2_cube.band("B03") * 0.0001
red = sentinel2_cube.band("B04") * 0.0001
nir = sentinel2_cube.band("B08") * 0.0001

# Cloud masking using SCL
# Mask out: water (6), cloud high prob (9), thin cirrus (10), snow/ice (11)
scl_band = sentinel2_cube.band("SCL")
mask = (scl_band == 6) | (scl_band == 9) | (scl_band == 10) | (scl_band == 11)
mask_resampled = mask.resample_cube_spatial(blue)

# Apply mask and create temporal composites (max value)
blue_composite = blue.mask(mask_resampled).max_time()
green_composite = green.mask(mask_resampled).max_time()
red_composite = red.mask(mask_resampled).max_time()
nir_composite = nir.mask(mask_resampled).max_time()

# Compute spectral indices
ndvi = (nir_composite - red_composite) / (nir_composite + red_composite)
savi = ((nir_composite - red_composite) / (nir_composite + red_composite + 0.428)) * 1.428

# Add band dimensions and merge
def add_band_dim(cube, name):
    return cube.add_dimension(name="bands", label=name, type="bands")

composites = [
    add_band_dim(blue_composite, "Blue"),
    add_band_dim(green_composite, "Green"),
    add_band_dim(red_composite, "Red"),
    add_band_dim(nir_composite, "NIR"),
    add_band_dim(ndvi, "NDVI"),
    add_band_dim(savi, "SAVI"),
]

merged_cube = composites[0]
for cube in composites[1:]:
    merged_cube = merged_cube.merge_cubes(cube)

# Execute and download
job = merged_cube.execute_batch()
job.get_results().download_files(OUTPUT_NAME)

---
## Part 2: Data Preparation

Load the 8-band composite (Sentinel-1 VV/VH + Sentinel-2 B2/B3/B4/B8 + NDVI + SAVI) and extract pixel values from labelled training polygons.

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask as rasterio_mask
import geopandas as gpd
from shapely.geometry import mapping
import matplotlib.pyplot as plt
from matplotlib import colors

from sklearn.preprocessing import QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    cross_val_score, cross_val_predict, GridSearchCV,
    StratifiedKFold, train_test_split
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score
)
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### Configuration

In [ ]:
MULTI_BAND_IMAGE = '2019_Jan_S1_S2_savi_ndvi_10m.tif'
LABELS_SHAPEFILE = '2019_January_32631_d.shp'
BAND_NAMES = ['VV', 'VH', 'B2', 'B3', 'B4', 'B8', 'NDVI', 'SAVI']
BANDS_TO_USE = [0, 1, 2, 3, 4, 5, 6]  # VV, VH, B2, B3, B4, B8, NDVI
FEATURE_NAMES = [BAND_NAMES[i] for i in BANDS_TO_USE]
CLASS_NAMES = ['Other', 'CDW']
TEST_SIZE = 0.35
CV_FOLDS = 15
RANDOM_STATE = 42
OUTPUT_DIR = 'results/'

In [ ]:
# Load multi-band raster
full_dataset = rasterio.open(MULTI_BAND_IMAGE)
n_bands = full_dataset.count
width = full_dataset.meta['width']
height = full_dataset.meta['height']

print(f'Image shape: {n_bands} bands x {height} rows x {width} cols')
print(f'CRS: {full_dataset.crs}')
print(f'Bands: {BAND_NAMES}')

In [ ]:
# Load training labels
shapefile = gpd.read_file(LABELS_SHAPEFILE)

print(f'Labels: {len(shapefile)} polygons')
print(f'Classes: {np.unique(shapefile.Classname)}')
print(f'CRS: {shapefile.crs}')
print(f'\nClass distribution:')
print(shapefile.Classname.value_counts())

In [ ]:
# Extract pixel values from labelled polygons
geoms = shapefile.geometry.values

X = np.array([], dtype=np.float32).reshape(n_bands, -1)
y = np.array([], dtype=str)

for index, geom in enumerate(geoms):
    feature = [mapping(geom)]
    classname = shapefile['Classname'][index]
    out_image = rasterio_mask(full_dataset, feature, nodata=10000, crop=True)[0]

    # Remove nodata pixels
    valid_mask = np.all(out_image != 10000, axis=0) & np.all(out_image != 0, axis=0)
    valid_pixels = out_image[:, valid_mask]

    if valid_pixels.size > 0:
        X = np.hstack([X, valid_pixels])
        y = np.append(y, [classname] * valid_pixels.shape[1])

# Transpose: (bands, pixels) -> (pixels, bands)
X = X.T
print(f'Extracted {X.shape[0]} pixels with {X.shape[1]} bands')
print(f'Labels: {np.unique(y, return_counts=True)}')

In [ ]:
# Prepare prediction array (full image)
X_pred = full_dataset.read()
X_pred = np.moveaxis(X_pred, 0, -1)
X_pred = X_pred.reshape(-1, full_dataset.meta['count'])

# Select bands
X = X[:, BANDS_TO_USE]
X_pred = X_pred[:, BANDS_TO_USE]

# Shuffle
rng = np.random.RandomState(RANDOM_STATE)
perm = rng.permutation(len(X))
X, y = X[perm], y[perm]

# Encode labels as integers
for i, label in enumerate(CLASS_NAMES):
    y[y == label] = i
y = y.astype(int)

print(f'Training features: {X.shape}')
print(f'Prediction features: {X_pred.shape}')
print(f'Bands used: {FEATURE_NAMES}')
print(f'\nLabel distribution:')
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {CLASS_NAMES[u]}: {c} pixels ({100*c/len(y):.1f}%)')

---
## Part 3: Model Training and Evaluation

In [ ]:
X_train, X_test, train_labels, test_labels = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

In [ ]:
def evaluate_model(name, clf, X_train, train_labels, X_test, test_labels, cv):
    """Train, evaluate, and return results for a classifier."""
    clf.fit(X_train, train_labels)
    cv_scores = cross_val_score(clf, X_train, train_labels, cv=cv)
    y_pred = cross_val_predict(clf, X_test, test_labels)
    f1 = f1_score(test_labels, y_pred, average='binary')
    test_acc = clf.score(X_test, test_labels)

    print(f'\n{"="*50}')
    print(f'{name}')
    print(f'{"="*50}')
    print(f'CV Accuracy:   {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')
    print(f'Test Accuracy: {test_acc:.4f}')
    print(f'F1 Score:      {f1:.4f}')
    print(f'\n{classification_report(test_labels, y_pred, target_names=CLASS_NAMES, digits=4)}')

    cm = confusion_matrix(test_labels, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f'TP={tp}, TN={tn}, FP={fp}, FN={fn}')
    print(f'False Positive Rate: {fp/(fp+tn):.4f}')
    print(f'False Negative Rate: {fn/(fn+tp):.4f}' if (fn+tp) > 0 else 'FNR: N/A (no positive samples)')

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues')
    ax.set_title(name)
    plt.tight_layout()
    plt.show()

    return {'name': name, 'clf': clf, 'cv_acc': cv_scores.mean(),
            'cv_std': cv_scores.std()*2, 'test_acc': test_acc,
            'f1': f1, 'y_pred': y_pred}

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS)
results = []

### 3.1 K-Nearest Neighbours (KNN)

In [ ]:
knn_pipe = make_pipeline(
    QuantileTransformer(n_quantiles=50, output_distribution='uniform'),
    KNeighborsClassifier()
)

knn_gs = GridSearchCV(
    estimator=knn_pipe,
    param_grid={
        'kneighborsclassifier__weights': ['uniform', 'distance'],
        'kneighborsclassifier__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
    },
    cv=CV_FOLDS, refit=True, verbose=0
)

res = evaluate_model('KNN', knn_gs, X_train, train_labels, X_test, test_labels, cv)
results.append(res)

### 3.2 Random Forest

In [ ]:
rf_pipe = make_pipeline(
    QuantileTransformer(n_quantiles=50, output_distribution='uniform'),
    RandomForestClassifier()
)

rf_gs = GridSearchCV(
    estimator=rf_pipe,
    param_grid={'randomforestclassifier__n_estimators': [100, 200]},
    cv=CV_FOLDS, refit=True, verbose=0
)

res = evaluate_model('Random Forest', rf_gs, X_train, train_labels, X_test, test_labels, cv)
results.append(res)

---
## Part 4: Model Comparison and Feature Importance

In [ ]:
# Compare all models
results_df = pd.DataFrame([
    {'Model': r['name'], 'CV Accuracy': r['cv_acc'], 'CV Std': r['cv_std'],
     'Test Accuracy': r['test_acc'], 'F1': r['f1']}
    for r in results
])

print('Model Comparison:')
print(results_df.sort_values('F1', ascending=False).to_string(index=False))

# Select best model by F1 score
best = max(results, key=lambda r: r['f1'])
best_clf = best['clf']
print(f'\nBest model: {best["name"]} (F1={best["f1"]:.4f})')

In [ ]:
# Permutation importance for the best model
perm_result = permutation_importance(
    best_clf, X_test, test_labels, n_repeats=10, random_state=RANDOM_STATE
)

importances = perm_result.importances_mean
sorted_idx = np.argsort(importances)[::-1]

print('Feature Importance (permutation):')
for idx in sorted_idx:
    print(f'  {FEATURE_NAMES[idx]:>6s}: {importances[idx]:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(importances)), importances[sorted_idx])
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([FEATURE_NAMES[i] for i in sorted_idx])
ax.set_ylabel('Permutation Importance')
ax.set_title(f'Feature Importance ({best["name"]})')
plt.tight_layout()
plt.show()

---
## Part 5: Full-Image Prediction and Export

In [ ]:
# Handle NaN values
X_pred_clean = np.nan_to_num(X_pred)

# Predict on full image
y_pred = best_clf.predict(X_pred_clean)
y_pred_img = y_pred.reshape((height, width))

print(f'Prediction shape: {y_pred_img.shape}')
unique, counts = np.unique(y_pred, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {CLASS_NAMES[u]}: {c} pixels ({100*c/len(y_pred):.1f}%)')

In [ ]:
# Visualise prediction
fig, ax = plt.subplots(figsize=(14, 10), dpi=100)
cmap = colors.ListedColormap(['lightgrey', 'red'])
im = ax.imshow(y_pred_img, cmap=cmap, interpolation='nearest')
ax.set_title(f'CDW Detection ({best["name"]})', fontsize=14)
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1], shrink=0.6)
cbar.ax.set_yticklabels(CLASS_NAMES)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Export prediction as GeoTIFF
os.makedirs(OUTPUT_DIR, exist_ok=True)

profile = full_dataset.profile.copy()
profile.update(dtype=rasterio.uint8, count=1, compress='lzw')

output_path = os.path.join(OUTPUT_DIR, f'CDW_prediction_{best["name"].replace(" ", "_")}.tif')
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(y_pred_img.astype(rasterio.uint8), 1)
print(f'Prediction saved to: {output_path}')

# Per-class binary masks
for i, label in enumerate(CLASS_NAMES):
    binary_mask = (y_pred_img == i).astype(rasterio.uint8)
    mask_path = os.path.join(OUTPUT_DIR, f'{label}_mask.tif')
    with rasterio.open(mask_path, 'w', **profile) as dst:
        dst.write(binary_mask, 1)
    print(f'Binary mask saved: {mask_path}')